# A Well-Balanced 🐴 + 🐰 Stew

<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-Two/blob/master/colab/Colab_The_Term_Structure_From_Par_Yields.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-Two/master?urlpath=lab/tree/notebooks/The_Term_Structure_From_Par_Yields.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-Two/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

This notebook combines the high-quality data of FRED with the Nelson-Siegel estimates anchored by the SOFR. Instead of going from the term structure to par yields, we are going from FRED par yields to the term structure.

The journey:

1. **The Starting Line (🐴):** The clean, highly curated FRED constant maturity data (Par Yields).
2. **Preparing The Data (🧑‍🍳):** Calculate payment data and payoff matrix.
2. **The Estimation (🐰):** Apply the constrained Nelson-Siegel model to estimate the spot rates.
3. **The Finish Line (🏁):** Estimate swap rates (Par Yields) and forward rates for arbitrary dates.

Our first step is to import some libraries and modules as well as functions from our custum module.


## Preparing the Notebook

<details style="border: 1px solid #b8daff; border-radius: 8px; padding: 15px; background-color: #f8fbff; margin: 20px 0;">
  <summary style="cursor: pointer; font-weight: bold; font-size: 1.1em; color: #004085; font-family: sans-serif; list-style-position: inside;">
    <span style="margin-right: 8px;">🛠️</span> Notebook Setup: Why the "Try/Except" Imports?
  </summary>
  <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #b8daff; color: #333333; font-family: sans-serif; font-size: 0.95em;">
    <b>The Goal:</b><br>
    To ensure this notebook runs perfectly whether you are using <b>Google Colab</b>, a local <b>Jupyter instance</b>, or a remote server without you having to manually install software.<br><br>
    <b>Key Concepts in this Section:</b>
    <ul style="line-height: 1.6; margin-bottom: 0;">
      <li><b>Standard Libraries:</b> Modules like <code>os</code>, <code>sys</code>, and <code>datetime</code> come "in the box" with Python. We use them for system tasks and date math.</li>
      <li><b>External Libraries:</b> NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.</li>
      <li><b>The <code>try/except</code> Logic:</b> This is a safety net.
        <ol style="margin-top: 4px; margin-bottom: 4px; padding-left: 20px;">
          <li>We <b>try</b> to import the library.</li>
          <li>If it fails (because it's not installed), the <b>except</b> block triggers a <code>!pip install</code> to download it automatically.</li>
        </ol>
      </li>
      <li><b>Aliasing (<code>as np</code>):</b> We rename <code>numpy</code> to <code>np</code> to save keystrokes. In professional finance code, <code>np</code> and <code>pd</code> are the universal shorthand.</li>
    </ul>
  </div>
</details>

### Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported.  (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
import os
import sys
import requests
from datetime import datetime, date
from types import ModuleType

try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

try:
    from scipy.optimize import minimize
except:
    !pip -q install scipy
    from scipy.optimize import minimize
```

In [ ]:
import os
import locale
import sys
import requests
from datetime import datetime, date
from types import ModuleType

try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

try:
    from scipy.optimize import minimize
except:
    !pip -q install scipy
    from scipy.optimize import minimize

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">👍 Cloud-Loading: How In-Memory Modules Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch:</b> <code>requests.get(url)</code> grabs the raw text of your Python script from Dropbox.</li>
<li><b>Instantiate:</b> <code>ModuleType(module_name)</code> creates an empty "container" in your computer's RAM.</li>
<li><b>Execute:</b> <code>exec(code, module.__dict__)</code> runs that text inside the container, turning text into live functions.</li>
<li><b>Register:</b> By adding it to <code>sys.modules</code>, we tell Python: <em>"If I try to import this later, don't look on the disk—look right here in the memory."</em></li>
</ol>

<p><b>Why do this?</b><br>
It makes your notebooks <b>100% portable</b>. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."</p>
</details>

In [ ]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.content.decode('utf-8'), module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (secure_key_setup,
                                             create_payoff_df,
                                             ns_spot_rates,
                                             estimate_ns_parameters,
                                             calc_par_yield,
                                              one_y_axis)

    print(f"✅ Successfully imported module from URL.")
    # Assign the FredReader attribute
    FredReader = module.FredReader
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")


## The Starting Line (🐴): FRED Constant Maturity Data

### ✅ Authenticate your FRED API key

Data is being accessed from the FRED database. The function `secure_key_setup` makes your FRED API key avilable in the notebook.  If you haven't yet add your key, the function will direct you to do so.

In [ ]:
secure_key_setup("fred_key")

### 📥 Fetching the Yield Curve Data
To build our curve, we first need the raw market data. Here, we instantiate our custom FredReader to fetch the constant maturity yields and overnight SOFR rate for our target settlement date of May 15, 2026.

Notice the built-in efficiency of the FredReader class: it seamlessly determines the environment (Colab or jupyter), securely retrieves the API key, and utilizes a local caching system. Because we have recently queried these series, the reader pulls the data directly from the local cache rather than making redundant API calls to FRED, significantly speeding up our workflow.



In [ ]:
# create an instance of FredReader
fred_data=FredReader()

# series ids
series_ids=['SOFR','DGS1MO','DGS3MO','DGS6MO',
         'DGS1','DGS2','DGS3','DGS5','DGS7','DGS10','DGS20','DGS30']

# the get_series method checks cache before accessing data from FRED
may_15_2026=fred_data.get_series(series_ids,start_date='2026-05-15',end_date='2026-05-15')

# show the results
display(may_15_2026)

### ♾️ Make the SOFR rate a continuously compounded actual/365 rate

For an overnight rate (where the actual days $d = 1$ for a standard weekday), the conversion is:
$$R_{c} = 365 \times \ln\left(1 + \frac{R_{SOFR}}{360}\right)$$

For a term SOFR rate over a period of $d$ days, the generalized conversion is:

$$R_{c} = \frac{365}{d} \times \ln\left(1 + R_{SOFR} \times \frac{d}{360}\right)$$
Where:
* $R_{c}$ = The new continuously compounded Actual/365 rate
* $R_{SOFR}$ = The published standard SOFR rate (expressed as a decimal, e.g., 0.053 for 5.3%)
* $\ln$ = Natural logarithm
* $d$ = The actual number of days in the period.

Because our settlement, May 15,2026, is a Friday, $d = 3$ and the continuous rate is:

$$R_{c} = \frac{365}{3} \times \ln\left(1 + R_{SOFR} \times \frac{3}{360}\right)$$

In [ ]:
# convert FRED sofr rate to discrete bond equivalent (365 vs 360 day year)
# May 15, 2026 to May 18,2026
d=3

# terminal value on May 18, 2026
terminal_value=(1+(may_15_2026['SOFR']/100).iloc[0]*3/360)

# make sofr rate continuous to match Nelson-Siegel estimates
sofr_rate=365/3*np.log(terminal_value)

display(f'Continuous-Actual/365 SOFR Rate:  {sofr_rate: .3%}')

## Preparing The Data (🧑‍🍳): Maturity Dates & Cash Flows

FRED's constant maturity yield data represents hypothetical par-value bonds issued on a specific settlement date. Because these yields are derived from the Federal Reserve's modeled term structure, we have to reverse-engineer the underlying cash flows of these hypothetical bonds to use them in our model. This data preparation pipeline involves:

* **Determining maturity dates**: Used to map out the exact schedule of future payment dates.

* **Deriving annual coupon rates**: Used to calculate the actual dollar amounts of each cash flow.

* **Constructing a cashflow payoff matrix**: Used as the mathematical foundation to price the bonds during each optimization iteration.

* **Setting benchmark prices**: Creating the target array of par values (100) for the hypothetical bonds.

#### 📅 Defining Settlement and Maturity Dates
We begin by establishing May 15, 2026, as our settlement date. Using this anchor, we project forward to calculate the exact calendar maturity dates for each instrument in our dataset. The time horizons (tenors) map directly to the FRED data, starting with a one-day offset for the overnight SOFR rate and extending through the standard constant maturity intervals up to 30 years.

In [ ]:
# assumed settlement date
settlement=date(2026,5,15)

# newly issued constant coupon maturity calculated from settlement
# SOFR overnight maturity, FRED maturity dates as reported
maturity_dates = pd.to_datetime(settlement) + np.array([
    pd.DateOffset(days=1),   pd.DateOffset(months=1), pd.DateOffset(months=3),
    pd.DateOffset(months=6), pd.DateOffset(years=1),  pd.DateOffset(years=2),
    pd.DateOffset(years=3),  pd.DateOffset(years=5),  pd.DateOffset(years=7),
    pd.DateOffset(years=10), pd.DateOffset(years=20), pd.DateOffset(years=30)
])

#### ✂️ Formatting the Coupon DataFrame
To prepare our data for the cash flow matrix, we need to extract and standardize the rates:

1. **Isolate the Data**: We extract the constant maturity yields for May 15, 2026.

2. **Scale Short-Term Rates**: We manually adjust the annualized SOFR, 1-month, and 3-month yields to equivalent semi-annual frequencies (accounting for SOFR's weekend accrual).

3. **Assemble the Target Matrix**: We combine the standardized coupons with our maturity_dates timeline to create the foundational DataFrame for our optimization.

In [ ]:
# extract the first row as a Series
coupons = may_15_2026.iloc[0].copy()

# adjust the short-term rates directly by their labels
coupons['SOFR'] *= 6 / 360
coupons['DGS1MO'] /= 6
coupons['DGS3MO'] /= 2

# create the DataFrame for creating payoff matrix
df = pd.DataFrame({'Coupon': coupons.values}, index=maturity_dates)

#### 💵 The Cash Flow Payoff Matrix
The `create_payoff_df` function has successfully generated our cash flow matrix. This DataFrame maps the exact dollar amount every hypothetical bond pays on every possible payment date across the entire 30-year curve.

**How to interpret this matrix**:

* **The Rows (Bonds)**: Each row represents one of our 12 standard instruments (from overnight SOFR to the 30-year Treasury).

* **The Columns (Dates)**: The columns represent every unique payment date across all instruments.

* **The Values (Cash Flows)**: The cells contain the projected cash flows.

    * Notice that the short-term instruments (Rows 0-3) show a single final payment (Principal + Interest) and zeros elsewhere.

    * The longer-term bonds (Rows 4-11) show recurring semi-annual coupon payments (e.g., \$2.56) leading up to a final maturity spike where the \$100 par principal is returned alongside the final coupon (e.g., $102.56).

Because most bonds do not pay on most dates, the resulting matrix is highly sparse (filled with zeros). This structural matrix is the mathematical foundation we will use to discount future cash flows and fit our spot rates.

#### 🚰 The Payoff Matrix: The Essential Plumbing

If we already knew all the present value (PV) discount factors, calculating the theoretical prices ($P$) of the bonds would be a simple matrix multiplication:

<br>

$$P = \text{Payoff Matrix} \times \text{PV Factors}$$

<br>

Those PV factors, however. are unknown. What we actually need to estimate are the spot rates of interest for the specific dates corresponding to the columns of our payoff matrix. The Nelson-Siegel model is the key 🔑 that unlocks this door. The restricted version of the Nelson-Siegel model defines the spot rate $r(t)$ as:

<br>

$$r(t) = \beta_0 + (r(0) - \beta_0)\left( \frac{1 - e^{-t/\tau}}{t/\tau} \right) + \beta_2 \left( \frac{1 - e^{-t/\tau}}{t/\tau} - e^{-t/\tau} \right)$$

<br>

🔄 As a reminder, our estimation process iterates on this model. We make continuous guesses for the values of $\beta_0$, $\beta_2$, and $\tau$, using them to generate spot rates (and thus our PV factors). We push those factors through the payoff matrix to get our forecasted prices, repeating this loop until the sum of squared differences between the forecasted prices and the actual market prices is minimized 🎯.

**💡 What, conceptually, are we doing here?**

 In the first Chapter of *The Term Structure Of Interst Rates* volume, we showed that the no-arbitrage condition implies that bond prices must equal their cash flows discounted at the corresponding spot rates. That is the exact principle driving this entire exercise!

#### 🎯 Setting the Target Benchmark Prices
In bond pricing, the dirty price represents the actual amount a buyer pays, which includes both the base market value (clean price) and any accrued interest owed to the seller. Because we are modeling newly issued hypothetical bonds on their exact settlement date, exactly zero days have passed since issuance. With no accrued interest to account for, the dirty price is simply equal to the bond's par value. We initialize this target price as exactly 100 for all 12 instruments.

In [ ]:
df['Dirty Price'] = np.full(12,100)

#### ⏱️🏷️ Preparing Inputs for Optimization: Time and Price
NumPy math handles Pandas objects seamlessly, strict optimization algorithms expect pure numerical arrays. To prepare our inputs:

* **Time to Payment (mat_years)**: We extract the column dates from the payoff DataFrame and convert them into a float array representing the fractional years since the settlement date.

* **Target Prices (P_actual)**: We isolate the 'Dirty Price' column to serve as our benchmark array. We will explicitly strip its Pandas index later during the Sum of Squared Residuals (SSR) calculation.

In [ ]:
# time to payment
mat_years=(pd.to_datetime(df_fred_payoff.columns)-pd.to_datetime('2026-05-15')).days/365.25

# target prices
P_actual_fred=df['Dirty Price']

## The Estimation (🐰): Apply the constrained Nelson-Siegel model to estimate the spot rates.

With our data prepped, we define the initial parameter guesses for our optimization routine. Because we are running a restricted model that forces the very short end of our curve to equal the overnight SOFR rate, the model only requires guesses for the long-term rate ($\beta_0$), the shape/curvature ($\beta_2$), and the decay factor ($\tau$). The slope parameter ($\beta_1$) is inherently constrained by the SOFR anchor. We pass these inputs to the optimizer to minimize the Sum of Squared Residuals (SSR) and extract our final spot rates.

In [ ]:
long_rate_guess=0.055
slope_guess=sofr_rate-long_rate_guess
shape_guess =0.0
tau_guess=2.0
guesses=[long_rate_guess,shape_guess,tau_guess]

# results include estiamted values and status of minimization problem
restricted=estimate_ns_parameters(df_fred_payoff,P_actual_fred,mat_years,guesses,sofr_rate=sofr_rate)

# calculate restricted spot rates
restricted_spot_rates, maturity_years = ns_spot_rates(restricted.x, mat_years,sofr_rate=sofr_rate)

### 🔢 Reviewing the Estimated Coefficients

* **Level ($\beta_0$)**: Our estimated long-term rate tracks very closely with the restricted FEDInvest estimate (5.68% versus 5.70%).
* **Short Rate ($\beta_0 + \beta_1$)**: Because we mathematically anchored the short end of the curve, the difference between the restricted short rates directly mirrors the difference in our long-term rates.
*  **Curvature ($\beta_2$)**: The shape parameter derived from the FRED data is approximately $\displaystyle \frac{1}{3}$ larger (less negative) than the FEDInvest counterpart (-0.014 versus -0.021).
* **Decay/Scaling ($\tau$)**: The restricted scaling factor is roughly 14% smaller than the FEDInvest baseline.

Although both datasets yield similar foundational estimates, our refined 🐴 + 🐰 methodology successfully dampens the extreme curvature and scaling values, ultimately generating a smoother, more stable spot rate curve.



### Plotting the Balanced 🐴 + 🐰 Stew
The proof is in the pudding (or in this case, the stew). Now that we have our optimized parameters, it is time to visualize the final product. By plotting our restricted zero-coupon spot rates across the full 30-year maturity spectrum, we can see exactly how our ingredients came together.

In [ ]:
plot_data=[restricted_spot_rates*100]
#X axis time in years
xaxis=maturity_years
series=['Restricted']
ylabel='Annual Percentage Rate'
xlabel='Maturity In Years'
size=(7,5)
markers=['']*1
colors=['b']

# use nanmin and nanmax so that nan is ignogred
y_lower=np.nanmin(plot_data)-1
y_upper=np.nanmax(plot_data)+1


ylim=[y_lower,y_upper]
title='SOFR Restricted Estimate Of Term Structure (FRED):'+settlement.strftime('%Y-%m-%d ')
one_y_axis(xaxis,plot_data,title,series,xlabel,ylabel,markers,size,ylim,
           colors=colors)

## **The Finish Line (🏁)**: Estimating Swap Rates (Par Yields) and Forward Rates for Arbitrary Dates
Now that we have a smooth, continuous curve, we are no longer restricted to standard maturity dates. We can interpolate and extract swap rates (par yields) and forward rates for any day in the future. Here is how we apply these tools to three classic benchmark scenarios:

1. **🔄 The Swap Rate (Par Yield)**: A corporation is considering an interest rate swap to hedge their borrowing costs for the exact duration of the next five years.

2. **⏭️ The Forward Borrowing Problem**: A corporation is attempting to project the exact financing costs of a large project they will not formally fund until the next fiscal year.

3. **⚖️ Creating An Investment Strategy**: Although the market believes that there will be no near-term changes in FED policy, an investor believes that the Federal Reserve will cut short rates in six to nine months and increase its sales of long-term bonds.



### ☝️📝 Before we begin, remember:

* **Par Yield Formula**:

    <blockquote>$\displaystyle\frac{1-PV(T)}{\sum _{t=1}^{T}PV(t)}$</blockquote><br>


* **Continuous Forward Rate Formula**:


  <blockquote>$\displaystyle\frac{\ln(PV(t))-\ln(PV(t+\Delta t))}{\Delta t}$</blockquote>



## 1. 🔄 The Interest Rate Swap

The Facts:

* **Principal Amount**: The corporation will borrow $100 million with an interest rate that resets quarterly.

* **Borrowing Costs**: In each quarter, the annualized borrowing rate will equal the annualized bond-equivalent yield (BEY) of 13-week Treasury bills at the start of each quarterly cycle, plus an annualized spread of 100 basis points or 1%.

* **The Hedging Instrument**: The corporation is evaluating a $100 million interest rate swap where they receive the exact BEY of a 13-week bill under the same terms as their borrowing agreement, transforming their floating-rate exposure into a predictable fixed cost.  The Corporation pays the agreed upon swap rate.

<img src='https://www.dropbox.com/scl/fi/zjscb5hm6mk0xh3evg26r/swap_graphic.png?rlkey=wt0lxx1aa6o3sq5hcnl97f63z&st=1i07b3j9&dl=1' width="600" height='300'>

### 1.1 📋 What do I need to know to help?
* When is the next time the interest rate will be reset?
    * **Reset Date**: Next settlement date May 15, 2026

* How long will the borrowing be in place?
    * **Expected Repayment Date**: May 14, 2031

* Exactly how long does the corporation wish to hedge its interest rate exposure?
    * **Expected Term Of Swap**: Two to five years

### 1.2 ⚙️ Calculating Swap Rates with `calc_par_yield` function

<details>

<summary style="cursor: pointer; font-size: 1.1em; color: #1a237e;"><b> 🔍 Click to see <code>calc_par_yields</code></b></summary>


```python
```python
def calc_par_yield(months_to_maturity, interim_estimates, sofr_rate=None, freq=6, forward_start_months=0):
    """
    Calculates spot or forward par yields.
    
    Args:
        months_to_maturity (int/float): Tenor in months. Must be an int if >= freq.
        interim_estimates (list): Nelson-Siegel parameters.
        sofr_rate (float, optional): Short rate for restricted model.
        freq (int, optional): Payment frequency in months. Defaults to 6.
        forward_start_months (int/float, optional): Forward start period. Defaults to 0 (spot).
    """
    # 1. Guard against negative maturities
    if months_to_maturity < 0:
        raise ValueError(f"Maturity cannot be negative. Received: {months_to_maturity}")
        
    # 2. Type check: Enforce integers ONLY for swap-market maturities (>= freq)
    if months_to_maturity >= freq and not isinstance(months_to_maturity, int):
        raise TypeError(
            f"months_to_maturity must be an integer when >= freq ({freq} months). "
            f"Received: {type(months_to_maturity).__name__} ({months_to_maturity})"
        )
    
    # 3. Safeguard against exactly 0 maturity (estimates the immediate curve intercept)
    months_to_maturity = max(months_to_maturity, 0.0001)
    
    # 4. Generate schedule of relative months and accrual periods (tau)
    if months_to_maturity < freq:
        # Short end: allows fractional months and single payments
        payment_months = [months_to_maturity]
        accrual_taus = [months_to_maturity / 12.0]
    else:
        # Long end: integer scheduling and sloppy stub handling
        full_periods = int(months_to_maturity // freq)
        payment_months = list(range(freq, (full_periods * freq) + 1, freq))
        accrual_taus = [freq / 12.0] * full_periods
        
        stub_months = months_to_maturity % freq
        if stub_months > 0:
            payment_months.append(months_to_maturity)
            accrual_taus.append(stub_months / 12.0)
            
    df = pd.DataFrame({
        'relative_months': payment_months,
        'tau': accrual_taus
    })
    
    # 5. Calculate Absolute Time (T) from TODAY (t=0)
    df['T_absolute'] = (df['relative_months'] + forward_start_months) / 12.0
    
    # 6. Get Spot Rates via your Nelson-Siegel function
    # if sofr_rate present check coeff
    if sofr_rate is not None and len(interim_estimates)>3:
      interim_estimates=list(interim_estimates)
      interim_estimates.pop(1)
    spot_rates, _ = ns_spot_rates(
        interim_estimates=interim_estimates,
        mat_years=df['T_absolute'].values,
        sofr_rate=sofr_rate
    )
    df['spot_rate'] = spot_rates
    
    # 7. Calculate Discount Factors back to today (t=0)
    df['DF'] = np.exp(-df['spot_rate'] * df['T_absolute'])
    
    # 8. Handle Forward Start Adjustment
    if forward_start_months > 0:
        T_start = forward_start_months / 12.0
        start_spot, _ = ns_spot_rates(interim_estimates, np.array([T_start]), sofr_rate)
        DF_start = np.exp(-start_spot[0] * T_start)
    else:
        DF_start = 1.0
        
    # 9. Calculate Par Yield
    DF_N = df['DF'].iloc[-1]
    PV01 = (df['tau'] * df['DF']).sum()
    
    par_yield = (DF_start - DF_N) / PV01
    
    return par_yield
```
    """
    num_periods = int(mat * freq)
    maturity = np.arange(1, num_periods + 1) / freq
    
    spot_rates, maturity = ns_spot_rates(coeff, maturity, sofr_rate)
    zeros = np.exp(-spot_rates * maturity)
    
    scale = 1 / maturity[-1] if maturity[-1] < 1 / freq else freq
    return (1 - zeros[-1]) / np.sum(zeros) * scale
```
</details>

**Arguments For Swap rates**
  * coeff: restricted estimates
  * freq: 4 (quarterly resets)
  * mat: 5 (five years) or 2 (two years)
  * sofr_rate: sofr_rate


In [ ]:
# restricted estimtes are assigned to coeff
coeff=restricted.x

# five year swap
mat=60

# quarterly reset dates
freq=3

# SOFR rate is for resricted estimtes
sofr_rate=sofr_rate

# five year swap rate
five_year_swap=calc_par_yield(mat,coeff,freq=freq,sofr_rate=sofr_rate)

# two year swap rate
mat=24

# two year swap rate
two_year_swap=calc_par_yield(mat,coeff,freq=freq,sofr_rate=sofr_rate)

display(f'Five Year Swap Rate: {five_year_swap: .2%}')
display(f'Two Year Swap Rate: {two_year_swap: .2%}')

<details open>
<summary style="font-size: 1.1em; font-weight: bold; cursor: pointer; margin-top: 15px;">✨ AI Study Assistant</summary>
<div style="padding: 15px; border: 1px solid #ddd; border-radius: 5px; margin-top: 10px; background-color: #fafafa;">

Select a prompt below to open Google AI and explore more concepts about the term sturcuture, par yields,and swap rates. *(Sign in to save your chat history!)*
---

✨ [**Can I estimate swap rates without the term structure? ↗**](https://www.google.com/search?udm=50&q=Can+I+estimate+swap+rates+without+the+term+structure?)

---

✨ [**How are par yields related to swap rates? ↗**](https://www.google.com/search?udm=50&q=How+are+par+yields+related+to+swap+rates?)---

---

✨ [**How are forward swap rates different for current swap rates? ↗**](https://www.google.com/search?udm=50&q=How+are+forward+swap+rates+different+from+current+swap+rates?)



<div style="padding: 15px; border: 1px solid #e0e0e0; border-left: 5px solid #ffc107; background-color: #fffde7; border-radius: 4px; margin-bottom: 15px;">
  <b> ✍️ Swap Rate Challenge.</b><br><br>

After reporting your two year and five year swap rates, the corporate CFO wants to know if it's better to enter a two-year swap today and a two- or three-year swap in two years.  

How would you help the CFO to explore this strategy?

<details>
<summary>🛟 <b>Need a solution?</b></summary>

<br>

💡 **The Solution: Analyze Forward Swap Rates**

To help the CFO make this decision, you need to calculate and present the **implied forward swap rates** (specifically, the 2-year forward 2-year, or 2-year forward 3-year swap rates).

By extracting these forward rates from today's spot yield curve, you can show the CFO exactly what the market is currently pricing in for a swap executed two years from now. The CFO can then compare those implied forward rates against their own internal forecasts to determine which strategy is most cost-effective.
</details>
</div>

## 2. **⏭️ Forecasting The Future Borrowing Rate**

There are two related ways to help the corporation forecast future borrowing rates.



1.   Calculate current foward rates of interest during the duration of the expected borrowing.
2.   Calculate the foward par yield or swap rate.



### 2.1 🔮 Glimpsing The Future With Forward Rates

Forward rates are the no arbitrage or certainty equivalent value of future spot rates-arguably the best anchor for forecasting spot rates.


As noted in Chapter Four of *The Term Structure Of Interest Rate Volume*, the formula for forward rates can be expressed as difference in spot rates and maturities:

 $$f(t,t+\Delta t)=\displaystyle\frac{\ln(PV(t))-\ln(PV(t+\Delta t))}{\Delta t} = \dfrac{-\text{spot}(t)\times t + \text{spot}(t+\Delta t)\times (t+\Delta t)}{\Delta t}$$

As demonstrated in *The Term Sturcture Of Interest Rate Volume*, forward rates are directly calculated with the Python code:



```
pot_rates, maturity_years = ns_spot_rates(results.x, mat_years)

# Create the 'spot_maturity' array with element-wise multiplication
spot_maturity = spot_rates * maturity_years
# np.diff calculates the difference between adjacent elements (i+1 - i).
# We divide the differences in spot_maturity by the differences in maturity_years.
forward_rates = np.diff(spot_maturity) / np.diff(maturity_years)
```

Because the coupon payment dates are in the future, we need to distinguish between the coupon payment dates and the current date.  The date that the funding time is assigned to `funding_date`.  In this example the funding time is assumed to be January 1, 2027 and the settlement date is May 15, 2026.

```
funding_date=(pd.to_datetime('2027-01-01')-pd.to_datetime('2026-05-15')).days/365.25
```
The duration of the borrowing is assumed to be ten years. The payment dates in years measured relative to the funding time is assigned to `payment years`.
```
payment_years=np.arange(1,11)
```
`maturity_years` is the array of both the `funding_date` and `payment_years`.
```
maturity_years=np.insert(funding_date+payment_years,0,funding_date)
```
`spot_rates` are calculated with the restricted Nelson-Siegel estimates:
```
spot_rates,maturity=ns_spot_rates(coeff,maturity,sofr_rate=sofr_rate)
```
Forward are then calculated for the ten borrowing horizion.
```
spot_maturity=spot_ratres*maturity
forward_rates=np.diff(spot_maturity)/np.diff(maturity)
```

### 2.2 🔮🔄 The Forecasted Coupon or Swap Rate

To calculate a forward swap or par yield, we need to recast the par yield formula to account for the start date.

* **Future Par Yield Formula**:


  <blockquote>$\displaystyle\frac{PV(\text{start)-PV(T)}}{\sum _{t>\text{start}}^{T}PV(t)}$</blockquote><br>

Dividing the numerator and denominator of the formula by the present value factor for the start date:
<br>
 $$PV(\text{start})$$
<br>
 This allows us to calculate par yields with forward rates.

* **Future Par Yield Formula Calculated With Forward Rates**

  <blockquote>$\displaystyle\frac{1-\frac{PV(T)}{PV(\text{start)}}}{\sum _{t>\text{start}}^{T}\frac{PV(t)}{PV(\text{start})}}=\displaystyle\frac{1-e^{-f(start,T)\times (T-start)}}{\sum _{t>\text{start}}^{T}e^{-f(start,t)\times (t-start)}}$</blockquote><br>

We first calculate forward rates relative to the start date of January 1, 2027:

```
# assumed maturity is 10 years
maturity=10
months_maturity=120
# funding at the turn of the year
funding_date=(pd.to_datetime('2027-01-01')-pd.to_datetime('2026-05-15'))/pd.Timedelta(days=30.436875)
```
The par yield is then calculated using these forward zero-coupon bond prices:
```
# assume a semi-annual bond is issued
freq=6

# the corporation estimates a required yield premium of 2%
credit_spread=0.02

# par yields of Treasury bonds

par_yield=calc_par_yield(months_maturity,coeff,sofr_rate=sofr_rate,
                         forward_start_months=funding_date)
```

This formulation allows us to calculate par yields or swap rates for the current or future dates.  Calculating forward rates relative to a start date arbitrarily close to zero (e.g. start = 1-e10) results in a rate for the current date.

The function `calc_par_yield` is used to estimate the ten-year par yield or swap rate the corporation could pay to receive the forward rate from the swap counterparty, or its annual coupon rate per \$100 of par value.

### 2.2 ⚙️ Calculating Swap Rates and Par Yields: `calc_par_yields` function



### 2.2.1 Estimating The Annual Coupon Rate Per \$100 of Par Value

* **Assumed spread above Treasury Bonds** 2.0%
* **Maturity**: ten years
* **Coupon Frequency**: semi annual

In [ ]:
# assumed maturity is 10 years
maturity=10
months_maturity=120
# funding at the turn of the year
funding_date=(pd.to_datetime('2027-01-01')-pd.to_datetime('2026-05-15'))/pd.Timedelta(days=30.436875)

# assume a semi-annual bond is issued
freq=6

# the corporation estimates a required yield premium of 2%
credit_spread=0.02

# par yields of Treasury bonds

par_yield=calc_par_yield(months_maturity,coeff,sofr_rate=sofr_rate,
                         forward_start_months=funding_date)

# corporate coupon yield including credit spread
annual_coupon=(par_yield+credit_spread)*100
display(f'Estimated Annual Coupon/$100 Par Value : ${annual_coupon:,.3f}')

## 3. **⚖️ Creating An Investment Strategy**

### 🧠 Test Your Knowledge of Forward And Swap Rates

**Assumptions**

1.  **Short-Term Rates**: You believe the FED will cut rates in the next six to nine months. The market anticipates no change.
2.  **The FED's Balance Sheet**: Coincident with the rate cut, you expect the FED to increase its sale of long-term bonds. The market anticipates no change.

Both of your beliefs predict a **steepening** term structure. You have designed a trading strategy that uses a ten-year interest rate swap and a six-month forward rate agreement that begins in six months.

Your forward rate agreement is a so-called 6x12 FRA. Like the interest rate swap, one party to the FRA pays a fixed amount and the other pays whatever the floating rate is over the six-month period.

#### 🤔 Questions

1. Should you be the fixed-rate payer or receiver of the 6x12 FRA?
2. Should you be the fixed-rate payer or receiver of the ten-year interest rate swap?

<details>
<summary><br>🔑 Answer Key / Explanation</br></summary>

* **💡 FRA**: Lock in the higher short-term rate to match your expectation of a cut in short rates. Pay the floating rate and **receive** the higher current fixed rate.
* **💡 Ten-Year Swap**: **Increased** sales of long-term bonds by the FED will cause long-term rates and ten-year **swap** rates to increase. If you are paying the lower fixed rate, the value of your position increases as rates rise. So, pay the fixed rate.

</details>

> **✍️ Update Swap and Forward Swap Rates**
>
> Using June 1, 2026 calculate:
> 1. The five-year swap rate beginning on June 1, 2026
> 2. The two-year swap rate beginning on June 1, 2026
> 3. The three-year swap rate beginning on June 1, 2029
> <br>

<details>
<summary><b>🛟 Need hints or a solution?</b></summary>
<p>Use June 1, 2026 constant maturity yields and the SOFR rate from FRED</p>

<blockquote>
<details>
<summary><b>💡 Hints</b></summary>
<p>Use June 1, 2026 constant maturity yields and the SOFR rate from FRED</p>
</details>

<details>
<summary><b>✅ Example Of Solution</b></summary>

### Example of Code

```python
# create an instance of FredReader
fred_data=FredReader()

# series ids
series_ids=['SOFR','DGS1MO','DGS3MO','DGS6MO',
         'DGS1','DGS2','DGS3','DGS5','DGS7','DGS10','DGS20','DGS30']

# fix date
start_end='2026-06-01'
# the get_series method checks cache before accessing data from FRED
june_1_2026=fred_data.get_series(series_ids,start_date=start_end,end_date=start_end)

# June 1, 2026 to June 2,2026
d=1

# terminal value on May 18, 2026
terminal_value=(1+(june_1_2026['SOFR']/100).iloc[0]*1/360)

# make sofr rate continuous to match Nelson-Siegel estimates
sofr_rate=365/1*np.log(terminal_value)

# extract the first row as a Series
coupons = june_1_2026.iloc[0].copy()

# adjust the short-term rates directly by their labels
coupons['SOFR'] *= 2 / 360
coupons['DGS1MO'] /= 6
coupons['DGS3MO'] /= 2

# create the DataFrame for creating payoff matrix
df = pd.DataFrame({'Coupon': coupons.values}, index=maturity_dates)

# create payoff matrix
df_fred_payoff=create_payoff_df(df,settlement,OLS=False)

# calculate dirty prices
df['Dirty Price'] = np.full(12,100)

# inputs for estimation
# time to payment
mat_years=(pd.to_datetime(df_fred_payoff.columns)-pd.to_datetime('2026-05-15')).days/365.25

# target prices
P_actual_fred=df['Dirty Price']

# restricted estimates of term structure June 1, 2026
long_rate_guess=0.055
slope_guess=sofr_rate-long_rate_guess
shape_guess =0.0
tau_guess=2.0
guesses=[long_rate_guess,shape_guess,tau_guess]

# results include estimated values and status of minimization problem
restricted=estimate_ns_parameters(df_fred_payoff,P_actual_fred,mat_years,guesses,sofr_rate=sofr_rate)

# calculate restricted spot rates
restricted_spot_rates, maturity_years = ns_spot_rates(restricted.x, mat_years,sofr_rate=sofr_rate)

# five year swap
mat=60
five_year_swap=calc_par_yield(mat,coeff,freq=freq,sofr_rate=sofr_rate)

# two year swap
mat=24
two_year_swap=calc_par_yield(mat,coeff,freq=freq,sofr_rate=sofr_rate)

# three year swap beginning in two years
months_maturity=36
three_year_forward_swap=calc_par_yield(months_maturity,coeff,sofr_rate=sofr_rate,forward_start_months=24)

# results
display(f'Five Year Swap Rate: {five_year_swap: .2%}')
display(f'Two Year Swap Rate: {two_year_swap: .2%}')
display(f'Three Year Forward Swap Rate: {three_year_forward_swap: .2%}')